In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
path = "/content/drive/MyDrive/AI&MED_Final/Data/"

files = {
    "DEMO":  "P_DEMO.xpt",
    "BMX":   "P_BMX.xpt",
    "BPX":   "P_BPXO.xpt",
    "HDL":   "P_HDL.xpt",
    "TRIGLY": "P_TRIGLY.xpt",
    "GLU":   "P_GLU.xpt",
}

tables = {}
for name, fname in files.items():
    df = pd.read_sas(path + fname, format="xport")
    tables[name] = df
    print(f"{name}: {df.shape[0]:,} rows × {df.shape[1]} cols")

In [ ]:
tables["DEMO"]   = tables["DEMO"][["SEQN","RIDAGEYR","RIAGENDR","RIDRETH3","RIDEXPRG"]]
tables["BMX"]    = tables["BMX"][["SEQN","BMXBMI","BMXWAIST","BMXHT"]]
tables["HDL"]    = tables["HDL"][["SEQN","LBDHDD"]]
tables["TRIGLY"] = tables["TRIGLY"][["SEQN","LBXTR"]]
tables["GLU"]    = tables["GLU"][["SEQN","LBXGLU"]]

bp = tables["BPX"]
sys_cols = [c for c in bp.columns if "SY" in c and c != "SEQN"]
dia_cols = [c for c in bp.columns if "DI" in c and c != "SEQN"]
tables["BPX"] = bp[["SEQN"] + sys_cols + dia_cols]

print("✓ Columns selected")
print(f"   BP systolic cols:  {sys_cols}")
print(f"   BP diastolic cols: {dia_cols}")

In [ ]:
from functools import reduce

df = reduce(
    lambda l, r: pd.merge(l, r, on="SEQN", how="inner"),
    tables.values()
)
print(f"✓ Merged: {df.shape[0]:,} rows × {df.shape[1]} cols")

In [ ]:
print(f"Start: {len(df):,}")

df = df[df["RIDAGEYR"] >= 18]
print(f"Age ≥ 18: {len(df):,}")

if "RIDEXPRG" in df.columns:
    df = df[df["RIDEXPRG"] != 1].drop(columns=["RIDEXPRG"])
    print(f"Non-pregnant: {len(df):,}")

sys_cols = [c for c in df.columns if "SY" in c]
dia_cols = [c for c in df.columns if "DI" in c]
df["avg_systolic"]  = df[sys_cols].mean(axis=1)
df["avg_diastolic"] = df[dia_cols].mean(axis=1)
df = df.drop(columns=sys_cols + dia_cols)

key = ["BMXBMI","BMXWAIST","BMXHT","avg_systolic",
       "avg_diastolic","LBDHDD","LBXTR","LBXGLU","RIDAGEYR","RIAGENDR"]
df = df.dropna(subset=key)
print(f"Complete cases: {len(df):,}")

In [ ]:
is_male   = df["RIAGENDR"] == 1
is_female = df["RIAGENDR"] == 2

df["c_waist"] = 0
df.loc[is_female & (df["BMXWAIST"] >= 88),  "c_waist"] = 1
df.loc[is_male   & (df["BMXWAIST"] >= 102), "c_waist"] = 1

df["c_trig"] = (df["LBXTR"] >= 150).astype(int)

df["c_hdl"] = 0
df.loc[is_female & (df["LBDHDD"] < 50), "c_hdl"] = 1
df.loc[is_male   & (df["LBDHDD"] < 40), "c_hdl"] = 1

df["c_bp"] = ((df["avg_systolic"] >= 130) | (df["avg_diastolic"] >= 85)).astype(int)

df["c_glu"] = (df["LBXGLU"] >= 100).astype(int)

crit = ["c_waist","c_trig","c_hdl","c_bp","c_glu"]
df["MetS"] = (df[crit].sum(axis=1) >= 3).astype(int)

print(f"MetS prevalence: {df['MetS'].mean()*100:.1f}%")
print(df["MetS"].value_counts())

In [ ]:
df = df.rename(columns={
    "RIDAGEYR": "age",
    "RIAGENDR": "sex",
    "RIDRETH3": "race_eth",
    "BMXBMI":   "bmi",
    "BMXWAIST": "waist_cm",
    "BMXHT":    "height_cm",
    "LBDHDD":   "hdl",
    "LBXTR":    "triglycerides",
    "LBXGLU":   "fasting_glucose",
})

df["sex"] = df["sex"].map({1: 1, 2: 0})

print("✓ Columns:", df.columns.tolist())

In [ ]:
print(df[["age","bmi","waist_cm","height_cm"]].describe().round(1))
print(f"\nSex split:\n{df['sex'].value_counts().rename({1:'Male',0:'Female'})}")
print(f"\nMetS by sex:\n{df.groupby('sex')['MetS'].mean().rename({1:'Male',0:'Female'}).round(3)}")

In [ ]:
out_path = "/content/drive/MyDrive/AI&MED_Final/Data/"

out = df[["SEQN","age","sex","race_eth","bmi","waist_cm","height_cm","MetS"]]
out.to_csv(out_path + "nhanes_clean.csv", index=False)
print(f"✓ Saved nhanes_clean.csv — {len(out):,} rows × {out.shape[1]} cols")

df.to_csv(out_path + "nhanes_clean_full.csv", index=False)
print(f"✓ Saved nhanes_clean_full.csv")